In [11]:
import pandas as pd

In [12]:
sos_total_og  = pd.read_excel("../data_storage/raw_data/voter_data/2024_total_gen_election.xlsx")
rdh_og = pd.read_csv("../data_storage/raw_data/voter_data/AL_l2_2024_gen_stats_2020county.csv")

In [13]:
sos_total  = sos_total_og.copy()
rdh = rdh_og.copy()

In [14]:
# SOS and RDH have different spacing/naming so needed to strip/rename columns. FIPS_code is a more standarized of naming.
#Used County to merge RDH and SoS later on in Cell 8 
sos_total["County"] = sos_total["County"].str.strip().str.replace("\n", " ").str.title()
rdh = rdh.rename(columns={"countyname": "County", "countyfips": "FIPS_code"})
rdh["County"] = rdh["County"].str.strip().str.title()

In [15]:
#Renames columns in different files to include SoS/RDH labeling.
sos_total  = sos_total.rename(columns={ "Total Ballots Cast": "SoS_total_voted","Registered Voters":  "SoS_total_reg"})
#Renames total ballots cast 
rdh = rdh.rename(columns={"voted_all": "RDH_total_voted","reg_all":   "RDH_total_reg"})

In [16]:
#Filters the 17 counties for Alabama BB
BLACK_BELT_COUNTIES = [ "Autauga", "Barbour", "Bullock", "Butler", "Choctaw","Clarke", "Conecuh", "Crenshaw", "Dallas", "Greene","Hale", "Lowndes", "Macon", "Marengo", "Monroe", "Perry", "Sumter"]
rdh_bb_filtered = rdh[rdh["County"].isin(BLACK_BELT_COUNTIES)].copy()

In [17]:
#This is an inner join because both the SoS and RDH have the same counties. Only gets the total columns and gender data from RDH.
rdh_sos_merged = sos_total.merge(rdh_bb_filtered[["County", "FIPS_code", "RDH_total_voted", "RDH_total_reg"]], on="County")

In [18]:
#Compares RDH and SoS total ballots cast and uses the absolute value to calculate the difference/percentage difference.
rdh_sos_merged["voted_difference"] = rdh_sos_merged["SoS_total_voted"] - rdh_sos_merged["RDH_total_voted"]
rdh_sos_merged["voted_abs_difference"] = rdh_sos_merged["voted_difference"].abs()
rdh_sos_merged["voted_difference_percent"] = rdh_sos_merged["voted_abs_difference"]/rdh_sos_merged["SoS_total_reg"]

In [19]:
#Compares RDH and SoS registration numbers and uses the absolute value to calculate the difference/percentage difference.
rdh_sos_merged["reg_difference"] = rdh_sos_merged["SoS_total_reg"] - rdh_sos_merged["RDH_total_reg"]
rdh_sos_merged["reg_abs_difference"] = rdh_sos_merged["reg_difference"].abs()
rdh_sos_merged["reg_difference_percent"] = rdh_sos_merged["reg_abs_difference"]/rdh_sos_merged["SoS_total_reg"]

In [20]:
#Displays the appropriate columns for total voting and registration across the two sources
rdh_sos_merged[["County", "FIPS_code", "SoS_total_voted", "RDH_total_voted", "voted_difference", "voted_difference_percent","SoS_total_reg","RDH_total_reg","reg_difference","reg_difference_percent"]].sort_values("County")

,County,FIPS_code,SoS_total_voted,RDH_total_voted,voted_difference,voted_difference_percent,SoS_total_reg,RDH_total_reg,reg_difference,reg_difference_percent
0,Autauga,1001,28388,28054,334,0.007215,46291,42772,3519,0.076019
1,Barbour,1005,9919,9745,174,0.009849,17666,16407,1259,0.071267
2,Bullock,1011,4144,4014,130,0.018103,7181,6850,331,0.046094
3,Butler,1013,8530,7945,585,0.040262,14530,13677,853,0.058706
4,Choctaw,1023,6692,6526,166,0.015417,10767,10136,631,0.058605
5,Clarke,1025,11993,11848,145,0.007575,19143,18286,857,0.044768
6,Conecuh,1035,6105,5463,642,0.065377,9820,9335,485,0.049389
7,Crenshaw,1041,6516,6392,124,0.011428,10851,10318,533,0.049120
8,Dallas,1047,15631,15451,180,0.006095,29534,27968,1566,0.053024
9,Greene,1063,4056,3968,88,0.013390,6572,6328,244,0.037127
